In [1]:
!python3.11 -m pip install langchain langchain-huggingface faiss-cpu datasets evaluate langchain-community -U
!python3.11 -m pip install torch accelerate transformers

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain
    Found existing installation: langchain 1.0.5
    Uninstalling langchain-1.0.5:
      Successfully uninstalled langchain-1.0.5
Defaulting to user installation because normal site-packages is not writeable


https://github.com/langchain-ai/docs/tree/main/src/oss/python/integrations/retrievers (ссылка на доки langchain по retriever'ам)

In [2]:
!python3.11 -V

Python 3.11.13


In [3]:
!python3.11 -m pip install langchain-core -U

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.0.4
    Uninstalling langchain-core-1.0.4:
      Successfully uninstalled langchain-core-1.0.4


In [4]:
!python3.11 -m pip install langchain-classic
!python3.11 -m pip install sentence-transformers
!python3.11 -m pip install rouge_score
!python3.11 -m pip install vllm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=29453c1c9aee36db2149acc248af17956466962a8ffd0c9877740c26c9708ca5
  Stored in directory: /home/ajalilov/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 35.7 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: outlines_core
    Found existing installation: outlines_cor

In [5]:
!python3.11 -m pip install rank_bm25

Defaulting to user installation because normal site-packages is not writeable


In [1]:
!pip install -U ipywidgets

     |████████████████████████████████| 139 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.2 MB 6.2 MB/s eta 0:00:01
     |████████████████████████████████| 914 kB 52.7 MB/s eta 0:00:01


In [18]:
from datasets import load_dataset
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains  import RetrievalQA 
from transformers import pipeline
import evaluate

In [19]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [20]:
from datasets import load_dataset
from typing import List, Tuple

def load_rag_benchmark_small() -> Tuple[List[str], List[str], List[str]]:
    """
    Загружает небольшой бенчмарк SQuAD для быстрого тестирования RAG системы.
    Около 100-200 примеров, быстрая загрузка, на английском языке.
    
    Returns:
        Tuple[List[str], List[str], List[str]]: contexts, questions, answers
    """
    
    # SQuAD - самый быстрый для загрузки и использования
    dataset = load_dataset("squad", split="validation[:200]")
    
    contexts = []
    questions = []
    answers = []
    
    for item in dataset:
        contexts.append(item['context'])
        questions.append(item['question'])
        answers.append(item['answers']['text'][0])
    
    return contexts, questions, answers


In [21]:
texts, questions, answers = load_rag_benchmark_small()


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 26eea470-8d86-4cd8-88bd-ec519bc8706a)')' thrown while requesting HEAD https://huggingface.co/datasets/squad/resolve/main/README.md
Retrying in 1s [Retry 1/5].


In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large-instruct")


In [ ]:
K = 3

In [ ]:
from langchain_core.documents import Document
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
documents = [Document(page_content=t) for t in texts]


# --- 1. Ваш семантический ретривер (FAISS) ---
print("Инициализация FAISS ретривера...")
db = FAISS.from_documents(documents, embeddings)
faiss_retriever = db.as_retriever(search_kwargs={"k": K}) # Увеличим k, чтобы дать ансамблю больше кандидатов


Инициализация FAISS ретривера...


In [8]:
# --- 2. Лексический ретривер (BM25) ---
print("Инициализация BM25 ретривера...")
# BM25Retriever.from_texts или .from_documents
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = K # Установим такое же k

Инициализация BM25 ретривера...


In [9]:
# --- 3. Создание ансамбля ---
print("Создание EnsembleRetriever...")
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]  # Можно настроить веса. 0.5/0.5 означает равную важность
)

Создание EnsembleRetriever...


In [10]:
from langchain_core.prompts import PromptTemplate

# создаём свой системный промпт
system_prompt = """
You must answer the question as clearly as possible based on the documents provided by user. In your answer, add a little thought and highlight your final answer as follows:
Final answer: "your answer". 
"""

# шаблон, который используется в RAG
prompt_template = """
Documents: {context}

Question: {question}

"""

# собираем кастомный шаблон, добавив системный инструктаж
full_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=system_prompt + "\n\n" + prompt_template,
)

In [11]:
os.environ["VLLM_ATTENTION_BACKEND"]="TORCH_SDPA"

In [12]:
from langchain_community.llms import VLLM

llm = VLLM(
    model="Qwen/Qwen3-VL-4B-Instruct",
    temperature=0,
    trust_remote_code=True,
    tensor_parallel_size=1,
    vllm_kwargs={
        "max_model_len": 4096,
        "max_num_batched_tokens": 4096,
        "gpu_memory_utilization": 0.7,
    },
)

INFO 11-16 11:35:34 [__init__.py:216] Automatically detected platform cuda.
WARNING 11-16 11:35:34 [cuda.py:619] Detected different devices in the system: NVIDIA GeForce RTX 2080 Ti, NVIDIA GeForce RTX 2080 Ti, NVIDIA A100-SXM4-80GB, NVIDIA A100-PCIE-40GB, NVIDIA A100-SXM4-80GB, NVIDIA A100-PCIE-40GB, NVIDIA GeForce RTX 2080 Ti, NVIDIA GeForce RTX 2080 Ti. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


INFO 11-16 11:35:36 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 4096, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-VL-4B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 11-16 11:35:37 [model.py:547] Resolved architecture: Qwen3VLForConditionalGeneration


`torch_dtype` is deprecated! Use `dtype` instead!


...

INFO 11-16 11:35:49 [__init__.py:216] Automatically detected platform cuda.
WARNING 11-16 11:35:49 [cuda.py:619] Detected different devices in the system: NVIDIA GeForce RTX 2080 Ti, NVIDIA GeForce RTX 2080 Ti, NVIDIA A100-SXM4-80GB, NVIDIA A100-PCIE-40GB, NVIDIA A100-SXM4-80GB, NVIDIA A100-PCIE-40GB, NVIDIA GeForce RTX 2080 Ti, NVIDIA GeForce RTX 2080 Ti. Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.
(EngineCore_DP0 pid=1097343) INFO 11-16 11:35:51 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1097343) INFO 11-16 11:35:51 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-VL-4B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen3-VL-4B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, d

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:06<00:06,  6.85s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:11<00:00,  5.78s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:11<00:00,  5.94s/it]
(EngineCore_DP0 pid=1097343) 


(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:14 [default_loader.py:267] Loading weights took 12.14 seconds
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:15 [gpu_model_runner.py:2653] Model loading took 8.6203 GiB and 13.072437 seconds
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:15 [gpu_model_runner.py:3344] Encoder cache will be initialized with a budget of 151250 tokens, and profiled with 1 video items of the maximum feature size.
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:38 [backends.py:548] Using cache directory: /home/ajalilov/.cache/vllm/torch_compile_cache/5dd02c67a5/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:38 [backends.py:559] Dynamo bytecode transform time: 7.97 s
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:43 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 3.186 s
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:45 [monitor.py:34] torch.compile takes 7.97 s in total
(EngineCore_D

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:04<00:00, 15.43it/s]


(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:53 [gpu_model_runner.py:3480] Graph capturing finished in 6 secs, took 0.62 GiB
(EngineCore_DP0 pid=1097343) INFO 11-16 11:36:53 [core.py:210] init engine (profile, create kv cache, warmup model) took 38.05 seconds
INFO 11-16 11:36:54 [llm.py:306] Supported_tasks: ['generate']


In [13]:
# 4️⃣ Сборка цепочки Retrieval-Augmented Generation
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type_kwargs={"prompt": full_prompt},
    retriever=ensemble_retriever,
    chain_type="stuff",
    return_source_documents=False
)


In [14]:
preds = []
for q in questions: 
    result = qa_chain.invoke(q)
    preds.append(result["result"])

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s, est. speed input: 879.66 toks/s, output: 59.16 toks/s]


In [15]:
import re

def extract_answer(text: str) -> str:
    """
    Извлекает содержимое кавычек после 'Final answer:' из строки.
    Пример:
        'Final answer: "В Древнем Египте"' → 'В Древнем Египте'
    """
    match = re.search(r'Final answer:\s*["“](.+?)["”]', text)
    return match.group(1) if match else text.strip()

In [16]:
res = []

for s in preds:
    res.append(extract_answer(s).lower())

answers_lower = [i.lower() for i in answers]

In [17]:
# 6️⃣ Оценка качества (ROUGE-L как быстрое приближение)
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=res, references=answers_lower)
print("ROUGE metrics:", results)

ROUGE metrics: {'rouge1': np.float64(0.6186709367270651), 'rouge2': np.float64(0.3877316079735152), 'rougeL': np.float64(0.6187423500974947), 'rougeLsum': np.float64(0.6181677402240993)}
